# deepseek_moe — pipeline

`deepseek-ai/deepseek-moe-16b-base` · GPU A100 40 GB · end-to-end ~7 h (cont 500 steps, align 1000 steps).

## 1. Setup — ~~6 min

In [ ]:
import os
os.environ['OLMOE_MOE_NAME'] = 'deepseek_moe'
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT     = '/content/drive/MyDrive/deepseek_moe-thesis'
DRIVE_MODELS   = f'{DRIVE_ROOT}/models'
DRIVE_RUNS     = f'{DRIVE_ROOT}/runs'
DRIVE_HF_CACHE = f'{DRIVE_ROOT}/hf_cache'
DRIVE_CODE     = f'{DRIVE_ROOT}/code'
REPO_DIR       = '/content/OLMOe'
REPO_URL       = ''

for p in (DRIVE_ROOT, DRIVE_MODELS, DRIVE_RUNS, DRIVE_HF_CACHE):
    os.makedirs(p, exist_ok=True)
os.environ['HF_HOME']            = DRIVE_HF_CACHE
os.environ['HF_DATASETS_CACHE']  = DRIVE_HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = DRIVE_HF_CACHE

In [ ]:
import shutil, subprocess
os.makedirs(REPO_DIR, exist_ok=True)
SOURCES = ('config.py', 'data_and_eval.py', 'models_and_training.py', 'routing_analysis.py', 'pyproject.toml')
copied = 0
for f in SOURCES:
    src = f'{DRIVE_CODE}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{REPO_DIR}/{f}')
        copied += 1
if copied == 0 and REPO_URL:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
    copied = sum(1 for f in SOURCES if os.path.exists(f'{REPO_DIR}/{f}'))
missing = [f for f in SOURCES[:4] if not os.path.exists(f'{REPO_DIR}/{f}')]
if missing:
    raise FileNotFoundError(f'Missing in {REPO_DIR}: {missing}. Upload to {DRIVE_CODE}/ or set REPO_URL.')
print(f'sources_copied={copied}/{len(SOURCES)}')

In [ ]:
%pip install -q 'transformers>=4.45' 'datasets>=2.18' 'accelerate>=0.30' 'huggingface_hub>=0.24' 'sacrebleu>=2.4' 'tensorboard>=2.15' matplotlib

In [ ]:
for local_name, drive_path in (('models', DRIVE_MODELS), ('runs', DRIVE_RUNS)):
    target = os.path.join(REPO_DIR, local_name)
    if os.path.islink(target):
        continue
    if os.path.exists(target):
        import shutil; shutil.rmtree(target)
    os.symlink(drive_path, target)

In [ ]:
import sys, torch, importlib, inspect
assert torch.cuda.is_available()
props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024**3
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
%cd $REPO_DIR
import config, data_and_eval, models_and_training, routing_analysis
for mod in (config, routing_analysis, models_and_training, data_and_eval):
    importlib.reload(mod)
need_gb = config.MOE_REGISTRY[config.MOE_NAME]['approx_vram_inference_gb']
assert total_gb + 1 >= need_gb, f'GPU {total_gb:.1f}GB < {need_gb}GB for {config.MOE_NAME}'
assert config.MOE_NAME == 'deepseek_moe'
assert config.TRUST_REMOTE_CODE is True
assert config.LEARNING_RATE_CONT  <= 1e-6
assert config.LEARNING_RATE_ALIGN <= 1e-6
assert config.WARMUP_STEPS >= 200
assert 'per_token' in inspect.signature(routing_analysis.routing_cache_to_sentence_distributions).parameters
assert 'pa.dim() == 3' in inspect.getsource(models_and_training.alignment_loss_fn)
print(f'GPU={props.name} {total_gb:.1f}GB  MOE={config.MOE_NAME}  ALIGN_LAYERS={config.ALIGN_LAYERS}')

In [ ]:
import importlib, models_and_training as m
importlib.reload(m)
_model, _tok = m.load_base_model(download_if_missing=True)
matches = sum(1 for _, mod in _model.named_modules() if mod.__class__.__name__ in config.ROUTER_CLASS_NAMES)
router_like = sorted({mod.__class__.__name__ for _, mod in _model.named_modules()
                      if 'router' in mod.__class__.__name__.lower() or 'gate' in mod.__class__.__name__.lower()})
print(f'router_class_matches={matches}  expected≈27  router_like={router_like}')
assert matches > 0
del _model, _tok
torch.cuda.empty_cache()

## 2. M1 baseline — ~~12 min

In [ ]:
import json, time, importlib, models_and_training as m
importlib.reload(m)
ckpt_dir = config.CHECKPOINT_DIR / 'model1_baseline'
ckpt_dir.mkdir(parents=True, exist_ok=True)
model, tokenizer = m.load_base_model(download_if_missing=True)
model.save_pretrained(str(ckpt_dir))
tokenizer.save_pretrained(str(ckpt_dir))
with (ckpt_dir / 'train_state.json').open('w', encoding='utf-8') as f:
    json.dump({'step': 0, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
               'init_source': config.BASE_MODEL_NAME, 'moe_name': config.MOE_NAME,
               'objective': 'baseline_snapshot_no_training'}, f, indent=2)
matches = sum(1 for _, mod in model.named_modules() if mod.__class__.__name__ in config.ROUTER_CLASS_NAMES)
print(f'snapshot={ckpt_dir}  router_class_matches={matches}')

In [ ]:
import torch
model.eval()
prompts = {'en': 'Luxembourg has three official languages:',
           'de': 'Luxemburg hat drei Amtssprachen:',
           'nl': 'Luxemburg heeft drie officiele talen:',
           'lu': 'Lëtzebuerg huet dräi offiziell Sproochen:'}
with torch.no_grad():
    for lang, prompt in prompts.items():
        enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        gen = model.generate(**enc, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        print(f'[{lang}]', tokenizer.decode(gen[0], skip_special_tokens=True))

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
import importlib, models_and_training as m; importlib.reload(m)
%time m.evaluate_model('baseline')
print(f'peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')

## 3. M2 cont — dry-run ~~4 min, full ~~1.25 h

In [ ]:
import shutil, torch, importlib, models_and_training as m
shutil.rmtree(config.CHECKPOINT_DIR / 'model2_cont', ignore_errors=True)
importlib.reload(m)
torch.cuda.reset_peak_memory_stats()
%time m.train_model_2_cont_pretrain(max_steps=50)
print(f'dry_run_peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')

In [ ]:
import shutil, torch, importlib, models_and_training as m
shutil.rmtree(config.CHECKPOINT_DIR / 'model2_cont', ignore_errors=True)
importlib.reload(m)
torch.cuda.reset_peak_memory_stats()
%time m.train_model_2_cont_pretrain(max_steps=500)
print(f'cont_train_peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')
torch.cuda.reset_peak_memory_stats()
%time m.evaluate_model('cont')
print(f'cont_eval_peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')

## 4. M3 align — dry-run ~~7 min, full ~~4.5 h

In [ ]:
import shutil, torch, importlib, models_and_training as m
shutil.rmtree(config.CHECKPOINT_DIR / 'model3_align', ignore_errors=True)
importlib.reload(m)
torch.cuda.reset_peak_memory_stats()
%time m.train_model_3_align(max_steps=50)
print(f'dry_run_peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')

In [ ]:
import shutil, torch, importlib, models_and_training as m
shutil.rmtree(config.CHECKPOINT_DIR / 'model3_align', ignore_errors=True)
importlib.reload(m)
torch.cuda.reset_peak_memory_stats()
%time m.train_model_3_align(max_steps=1000)
print(f'align_train_peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')
torch.cuda.reset_peak_memory_stats()
%time m.evaluate_model('align')
print(f'align_eval_peak_vram_gb={torch.cuda.max_memory_allocated()/1024**3:.3f}')

## 5. 3-stage table

In [ ]:
import json
def jload(p):
    with open(p) as f: return json.load(f)
rows = [(st, jload(str(config.EVAL_DIR / f'eval_{st}.json'))) for st in ('baseline','cont','align')]
print(f"{'stage':>9} | {'ppl_en':>7} | {'ppl_de':>7} | {'ppl_nl':>7} | {'ppl_lu':>7} | {'bleu':>7} | {'chrf':>7} | {'n':>4}")
for st, d in rows:
    p, l = d['ppl'], d['luxgen']
    print(f"{st:>9} | {p['en']:7.3f} | {p['de']:7.3f} | {p['nl']:7.3f} | {p['lu']:7.3f} | {l.get('bleu',0):7.3f} | {l.get('chrf',0):7.3f} | {l.get('num_samples'):>4}")
c, a = rows[1][1], rows[2][1]
dppl = a['ppl']['lu'] - c['ppl']['lu']
dbleu = a['luxgen'].get('bleu',0) - c['luxgen'].get('bleu',0)
n = a['luxgen'].get('num_samples',0)
verdict = 'HELPS' if (dppl < -0.5 or (dbleu > 1.0 and n >= 30)) else ('HURTS' if dppl > 0.5 else 'inconclusive')
print(f'\nΔppl_lu={dppl:+.3f}  Δbleu={dbleu:+.3f}  n={n}  →  {verdict}')

## 6. Multi-seed eval — ~~1 h

In [ ]:
import importlib, models_and_training as m; importlib.reload(m)
%time m.run_multi_seed_evaluation_suite(eval_model='all', seeds=[42, 43, 44], include_routing=True)

In [ ]:
import csv
with (config.EVAL_DIR / 'comparison_metrics_multiseed_aggregate.csv').open() as f:
    for row in csv.reader(f):
        print(row)

## 7. Cross-arch

After all 3 archs done: `python colab/final_results.py`.